# Title and Image Similarity Score - Adding in Dataset

## Cosine Similarity

In [ ]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from torch.nn.functional import cosine_similarity
from tqdm import tqdm

# Load data
df = pd.read_csv('/content/drive/MyDrive/Research Important Datasets/Final/finalDataset.csv')

# Fill missing flags
df['image_exists'] = df['image_exists'].fillna('FALSE')

# Load BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')
bert_model.eval()
bert_model.to('cuda' if torch.cuda.is_available() else 'cpu')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Function to get BERT CLS vector
def get_bert_vector(text):
    encoded = tokenizer(text, return_tensors='pt', truncation=True, padding='max_length', max_length=32)
    encoded = {key: val.to(device) for key, val in encoded.items()}
    with torch.no_grad():
        output = bert_model(**encoded)
        cls_vector = output.last_hidden_state[:, 0, :]  # CLS token
    return cls_vector.squeeze(0)

# Compute similarity
similarities = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Computing similarities"):
    title = str(row['title'])
    image_exists = str(row['image_exists']).strip().upper()
    caption = str(row['caption']) if not pd.isna(row['caption']) else ""

    if image_exists == "TRUE" and caption.strip() != "":
        try:
            title_vec = get_bert_vector(title)
            caption_vec = get_bert_vector(caption)
            sim_score = cosine_similarity(title_vec.unsqueeze(0), caption_vec.unsqueeze(0)).item()
        except Exception as e:
            print(f"Error at index {idx}: {e}")
            sim_score = 0.0
    else:
        sim_score = 0.0

    similarities.append(sim_score)

# Add to dataframe
df['image_caption_similarity'] = similarities

# Save to CSV
output_path = '/content/drive/MyDrive/Research Important Datasets/Final/finalDataset_cosine.csv'
df.to_csv(output_path, index=False)

print(f"\n✅ Saved updated dataset with 'image_caption_similarity' to:\n{output_path}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Computing similarities:  57%|█████▋    | 57215/100000 [3:29:12<2:32:19,  4.68it/s]

## Sentence-BERT + Cosine Similarity

In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

# Load data
df = pd.read_csv('/content/drive/MyDrive/Research Important Datasets/50kimage_generate.csv')

# Fill missing flags
df['image_exists'] = df['image_exists'].fillna('FALSE')

# Load SBERT model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
sbert_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# Compute similarity
similarities = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Computing SBERT similarities"):
    title = str(row['title'])
    image_exists = str(row['image_exists']).strip().upper()
    caption = str(row['caption']) if not pd.isna(row['caption']) else ""

    if image_exists == "TRUE" and caption.strip() != "":
        try:
            title_vec = sbert_model.encode(title, convert_to_tensor=True)
            caption_vec = sbert_model.encode(caption, convert_to_tensor=True)
            sim_score = util.pytorch_cos_sim(title_vec, caption_vec).item()
        except Exception as e:
            print(f"Error at index {idx}: {e}")
            sim_score = 0.0
    else:
        sim_score = 0.0

    similarities.append(sim_score)

# Add to dataframe
df['image_caption_similarity'] = similarities

# Save updated dataset
output_path = '/content/drive/MyDrive/Research Important Datasets/Final_with_caption_similarity_50k_cbert.csv'
df.to_csv(output_path, index=False)

print(f"\n✅ Saved updated dataset with SBERT similarity to:\n{output_path}")

/tmp/ipython-input-3-1642442547.py:7: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/content/drive/MyDrive/Research Important Datasets/50kimage_generate.csv')
Computing SBERT similarities: 100%|██████████| 100000/100000 [08:08<00:00, 204.57it/s]



✅ Saved updated dataset with SBERT similarity to:
/content/drive/MyDrive/Research Important Datasets/Final_with_caption_similarity_50k_cbert.csv


# Model Trained - Checking Accuracy

## Cosine Similarity

In [ ]:
# Hybrid BERT + Numeric + Region Embedding
import pandas as pd
import numpy as np
import torch
import time
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

# ================== LOAD & PREPROCESS DATA ==================
# Load dataset
df = pd.read_csv('/content/drive/MyDrive/Research Important Datasets/Final_with_caption_similarity_50k_cosine.csv')
df = df.dropna(subset=['title'])

# Fill missing values
numeric_cols = ['num_comments', 'score', 'upvote_ratio', 'polarity', 'emotion_score', 'title_dup_count', 'image_caption_similarity']
df[numeric_cols] = df[numeric_cols].fillna(0)
df['sub_region'] = df['sub_region'].fillna('unknown')

# Region ID encoding
df['region_id'] = df['sub_region'].astype('category').cat.codes
region2id = dict(enumerate(df['sub_region'].astype('category').cat.categories))
num_regions = len(region2id)

# Final feature sets
all_numeric = numeric_cols
X_text = df['title'].values
X_numeric = df[all_numeric].values
X_region = df['region_id'].values
y = df['2_way_label'].values

# ================== SPLIT DATA ==================
X_text_tv, X_text_test, X_num_tv, X_num_test, X_reg_tv, X_reg_test, y_tv, y_test = train_test_split(
    X_text, X_numeric, X_region, y, test_size=0.15, stratify=y, random_state=42
)
X_text_train, X_text_val, X_num_train, X_num_val, X_reg_train, X_reg_val, y_train, y_val = train_test_split(
    X_text_tv, X_num_tv, X_reg_tv, y_tv, test_size=0.1765, stratify=y_tv, random_state=42
)

# Normalize numeric features
scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_num_train)
X_num_val = scaler.transform(X_num_val)
X_num_test = scaler.transform(X_num_test)

# ================== TOKENIZER ==================
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# ================== CUSTOM DATASET ==================
class FakeNewsDataset(Dataset):
    def __init__(self, texts, numerics, region_ids, labels):
        self.texts = texts
        self.numerics = numerics
        self.region_ids = region_ids
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=32,
            return_tensors="pt"
        )
        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'numerics': torch.tensor(self.numerics[idx], dtype=torch.float32),
            'region_id': torch.tensor(self.region_ids[idx], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# ================== DATALOADERS ==================
train_dataset = FakeNewsDataset(X_text_train, X_num_train, X_reg_train, y_train)
val_dataset   = FakeNewsDataset(X_text_val, X_num_val, X_reg_val, y_val)
test_dataset  = FakeNewsDataset(X_text_test, X_num_test, X_reg_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64)
test_loader  = DataLoader(test_dataset, batch_size=64)

# ================== MODEL ==================
class HybridBERTRegionEmbeddingModel(nn.Module):
    def __init__(self, numeric_input_dim, num_regions, region_embed_dim=16):
        super(HybridBERTRegionEmbeddingModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.region_embedding = nn.Embedding(num_regions, region_embed_dim)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(768 + numeric_input_dim + region_embed_dim, 128)
        self.fc2 = nn.Linear(128, 1)
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask, numerics, region_ids):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        region_embeds = self.region_embedding(region_ids)
        combined = torch.cat((cls_output, numerics, region_embeds), dim=1)
        x = self.relu(self.fc1(self.dropout(combined)))
        return torch.sigmoid(self.fc2(x))

# ================== TRAINING ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridBERTRegionEmbeddingModel(
    numeric_input_dim=X_num_train.shape[1],
    num_regions=num_regions
).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
start_time = time.time()

for epoch in range(3):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        numerics = batch['numerics'].to(device)
        region_ids = batch['region_id'].to(device)
        labels = batch['label'].to(device).unsqueeze(1)

        outputs = model(input_ids, attention_mask, numerics, region_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")

end_time = time.time()
print(f"\nTotal training time: {end_time - start_time:.2f} seconds")

# ================== EVALUATION ==================
def evaluate_model(dataloader, dataset_name="Validation"):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            numerics = batch['numerics'].to(device)
            region_ids = batch['region_id'].to(device)
            labels = batch['label'].to(device).unsqueeze(1)

            outputs = model(input_ids, attention_mask, numerics, region_ids)
            probs = outputs.cpu().numpy()
            preds = (probs > 0.5).astype(int)

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)

    print(f"\n{dataset_name} Classification Report:")
    print(classification_report(all_labels, all_preds))
    if np.isnan(all_probs).any():
        print(f"Warning: NaNs in {dataset_name} probabilities. Skipping AUC.")
    else:
        auc_score = roc_auc_score(all_labels, all_probs)
        print(f"{dataset_name} AUC: {auc_score:.4f}")

# Evaluate
evaluate_model(val_loader, "Validation")
evaluate_model(test_loader, "Test")

Epoch 1 Loss: 307.6698
Epoch 2 Loss: 216.1064
Epoch 3 Loss: 143.5332

Total training time: 1250.52 seconds

Validation Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.85      0.88      6377
         1.0       0.90      0.94      0.92      8626

    accuracy                           0.90     15003
   macro avg       0.91      0.90      0.90     15003
weighted avg       0.91      0.90      0.90     15003

Validation AUC: 0.9655

Test Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.86      0.89      6375
         1.0       0.90      0.95      0.92      8625

    accuracy                           0.91     15000
   macro avg       0.91      0.90      0.90     15000
weighted avg       0.91      0.91      0.91     15000

Test AUC: 0.9662


## Sentence Bert + Cosine Similarity

In [ ]:
# Hybrid BERT + Numeric + Region Embedding
import pandas as pd
import numpy as np
import torch
import time
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

# ================== LOAD & PREPROCESS DATA ==================
# Load dataset
df = pd.read_csv('/content/drive/MyDrive/Research Important Datasets/Final_with_caption_similarity_50k_cbert.csv')
df = df.dropna(subset=['title'])

# Fill missing values
numeric_cols = ['num_comments', 'score', 'upvote_ratio', 'polarity', 'emotion_score', 'title_dup_count', 'image_caption_similarity']
df[numeric_cols] = df[numeric_cols].fillna(0)
df['sub_region'] = df['sub_region'].fillna('unknown')

# Region ID encoding
df['region_id'] = df['sub_region'].astype('category').cat.codes
region2id = dict(enumerate(df['sub_region'].astype('category').cat.categories))
num_regions = len(region2id)

# Final feature sets
all_numeric = numeric_cols
X_text = df['title'].values
X_numeric = df[all_numeric].values
X_region = df['region_id'].values
y = df['2_way_label'].values

# ================== SPLIT DATA ==================
X_text_tv, X_text_test, X_num_tv, X_num_test, X_reg_tv, X_reg_test, y_tv, y_test = train_test_split(
    X_text, X_numeric, X_region, y, test_size=0.15, stratify=y, random_state=42
)
X_text_train, X_text_val, X_num_train, X_num_val, X_reg_train, X_reg_val, y_train, y_val = train_test_split(
    X_text_tv, X_num_tv, X_reg_tv, y_tv, test_size=0.1765, stratify=y_tv, random_state=42
)

# Normalize numeric features
scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_num_train)
X_num_val = scaler.transform(X_num_val)
X_num_test = scaler.transform(X_num_test)

# ================== TOKENIZER ==================
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# ================== CUSTOM DATASET ==================
class FakeNewsDataset(Dataset):
    def __init__(self, texts, numerics, region_ids, labels):
        self.texts = texts
        self.numerics = numerics
        self.region_ids = region_ids
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=32,
            return_tensors="pt"
        )
        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'numerics': torch.tensor(self.numerics[idx], dtype=torch.float32),
            'region_id': torch.tensor(self.region_ids[idx], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# ================== DATALOADERS ==================
train_dataset = FakeNewsDataset(X_text_train, X_num_train, X_reg_train, y_train)
val_dataset   = FakeNewsDataset(X_text_val, X_num_val, X_reg_val, y_val)
test_dataset  = FakeNewsDataset(X_text_test, X_num_test, X_reg_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64)
test_loader  = DataLoader(test_dataset, batch_size=64)

# ================== MODEL ==================
class HybridBERTRegionEmbeddingModel(nn.Module):
    def __init__(self, numeric_input_dim, num_regions, region_embed_dim=16):
        super(HybridBERTRegionEmbeddingModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.region_embedding = nn.Embedding(num_regions, region_embed_dim)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(768 + numeric_input_dim + region_embed_dim, 128)
        self.fc2 = nn.Linear(128, 1)
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask, numerics, region_ids):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        region_embeds = self.region_embedding(region_ids)
        combined = torch.cat((cls_output, numerics, region_embeds), dim=1)
        x = self.relu(self.fc1(self.dropout(combined)))
        return torch.sigmoid(self.fc2(x))

# ================== TRAINING ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridBERTRegionEmbeddingModel(
    numeric_input_dim=X_num_train.shape[1],
    num_regions=num_regions
).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
start_time = time.time()

for epoch in range(3):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        numerics = batch['numerics'].to(device)
        region_ids = batch['region_id'].to(device)
        labels = batch['label'].to(device).unsqueeze(1)

        outputs = model(input_ids, attention_mask, numerics, region_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")

end_time = time.time()
print(f"\nTotal training time: {end_time - start_time:.2f} seconds")

# ================== EVALUATION ==================
def evaluate_model(dataloader, dataset_name="Validation"):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            numerics = batch['numerics'].to(device)
            region_ids = batch['region_id'].to(device)
            labels = batch['label'].to(device).unsqueeze(1)

            outputs = model(input_ids, attention_mask, numerics, region_ids)
            probs = outputs.cpu().numpy()
            preds = (probs > 0.5).astype(int)

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)

    print(f"\n{dataset_name} Classification Report:")
    print(classification_report(all_labels, all_preds))
    if np.isnan(all_probs).any():
        print(f"Warning: NaNs in {dataset_name} probabilities. Skipping AUC.")
    else:
        auc_score = roc_auc_score(all_labels, all_probs)
        print(f"{dataset_name} AUC: {auc_score:.4f}")

# Evaluate
evaluate_model(val_loader, "Validation")
evaluate_model(test_loader, "Test")

Epoch 1 Loss: 307.7340
Epoch 2 Loss: 216.9172
Epoch 3 Loss: 142.1818

Total training time: 1247.53 seconds

Validation Classification Report:
              precision    recall  f1-score   support

         0.0       0.90      0.87      0.89      6377
         1.0       0.91      0.93      0.92      8626

    accuracy                           0.90     15003
   macro avg       0.90      0.90      0.90     15003
weighted avg       0.90      0.90      0.90     15003

Validation AUC: 0.9644

Test Classification Report:
              precision    recall  f1-score   support

         0.0       0.90      0.88      0.89      6375
         1.0       0.91      0.93      0.92      8625

    accuracy                           0.91     15000
   macro avg       0.91      0.90      0.90     15000
weighted avg       0.91      0.91      0.91     15000

Test AUC: 0.9648


## Without Title Duplicate Count - CBERT Similarity

In [ ]:
# Hybrid BERT + Numeric + Region Embedding
import pandas as pd
import numpy as np
import torch
import time
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

# ================== LOAD & PREPROCESS DATA ==================
# Load dataset
df = pd.read_csv('/content/drive/MyDrive/Research Important Datasets/Final_with_caption_similarity_50k_cbert.csv')
df = df.dropna(subset=['title'])

# Fill missing values
numeric_cols = ['num_comments', 'score', 'upvote_ratio', 'polarity', 'emotion_score', 'image_caption_similarity']
df[numeric_cols] = df[numeric_cols].fillna(0)
df['sub_region'] = df['sub_region'].fillna('unknown')

# Region ID encoding
df['region_id'] = df['sub_region'].astype('category').cat.codes
region2id = dict(enumerate(df['sub_region'].astype('category').cat.categories))
num_regions = len(region2id)

# Final feature sets
all_numeric = numeric_cols
X_text = df['title'].values
X_numeric = df[all_numeric].values
X_region = df['region_id'].values
y = df['2_way_label'].values

# ================== SPLIT DATA ==================
X_text_tv, X_text_test, X_num_tv, X_num_test, X_reg_tv, X_reg_test, y_tv, y_test = train_test_split(
    X_text, X_numeric, X_region, y, test_size=0.15, stratify=y, random_state=42
)
X_text_train, X_text_val, X_num_train, X_num_val, X_reg_train, X_reg_val, y_train, y_val = train_test_split(
    X_text_tv, X_num_tv, X_reg_tv, y_tv, test_size=0.1765, stratify=y_tv, random_state=42
)

# Normalize numeric features
scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_num_train)
X_num_val = scaler.transform(X_num_val)
X_num_test = scaler.transform(X_num_test)

# ================== TOKENIZER ==================
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# ================== CUSTOM DATASET ==================
class FakeNewsDataset(Dataset):
    def __init__(self, texts, numerics, region_ids, labels):
        self.texts = texts
        self.numerics = numerics
        self.region_ids = region_ids
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=32,
            return_tensors="pt"
        )
        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'numerics': torch.tensor(self.numerics[idx], dtype=torch.float32),
            'region_id': torch.tensor(self.region_ids[idx], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# ================== DATALOADERS ==================
train_dataset = FakeNewsDataset(X_text_train, X_num_train, X_reg_train, y_train)
val_dataset   = FakeNewsDataset(X_text_val, X_num_val, X_reg_val, y_val)
test_dataset  = FakeNewsDataset(X_text_test, X_num_test, X_reg_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64)
test_loader  = DataLoader(test_dataset, batch_size=64)

# ================== MODEL ==================
class HybridBERTRegionEmbeddingModel(nn.Module):
    def __init__(self, numeric_input_dim, num_regions, region_embed_dim=16):
        super(HybridBERTRegionEmbeddingModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.region_embedding = nn.Embedding(num_regions, region_embed_dim)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(768 + numeric_input_dim + region_embed_dim, 128)
        self.fc2 = nn.Linear(128, 1)
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask, numerics, region_ids):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        region_embeds = self.region_embedding(region_ids)
        combined = torch.cat((cls_output, numerics, region_embeds), dim=1)
        x = self.relu(self.fc1(self.dropout(combined)))
        return torch.sigmoid(self.fc2(x))

# ================== TRAINING ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridBERTRegionEmbeddingModel(
    numeric_input_dim=X_num_train.shape[1],
    num_regions=num_regions
).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
start_time = time.time()

for epoch in range(3):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        numerics = batch['numerics'].to(device)
        region_ids = batch['region_id'].to(device)
        labels = batch['label'].to(device).unsqueeze(1)

        outputs = model(input_ids, attention_mask, numerics, region_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")

end_time = time.time()
print(f"\nTotal training time: {end_time - start_time:.2f} seconds")

# ================== EVALUATION ==================
def evaluate_model(dataloader, dataset_name="Validation"):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            numerics = batch['numerics'].to(device)
            region_ids = batch['region_id'].to(device)
            labels = batch['label'].to(device).unsqueeze(1)

            outputs = model(input_ids, attention_mask, numerics, region_ids)
            probs = outputs.cpu().numpy()
            preds = (probs > 0.5).astype(int)

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)

    print(f"\n{dataset_name} Classification Report:")
    print(classification_report(all_labels, all_preds))
    if np.isnan(all_probs).any():
        print(f"Warning: NaNs in {dataset_name} probabilities. Skipping AUC.")
    else:
        auc_score = roc_auc_score(all_labels, all_probs)
        print(f"{dataset_name} AUC: {auc_score:.4f}")

# Evaluate
evaluate_model(val_loader, "Validation")
evaluate_model(test_loader, "Test")

Epoch 1 Loss: 310.0342
Epoch 2 Loss: 213.6973
Epoch 3 Loss: 139.0645

Total training time: 1255.11 seconds

Validation Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.85      0.89      6377
         1.0       0.90      0.94      0.92      8626

    accuracy                           0.91     15003
   macro avg       0.91      0.90      0.90     15003
weighted avg       0.91      0.91      0.91     15003

Validation AUC: 0.9651

Test Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.86      0.89      6375
         1.0       0.90      0.94      0.92      8625

    accuracy                           0.91     15000
   macro avg       0.91      0.90      0.91     15000
weighted avg       0.91      0.91      0.91     15000

Test AUC: 0.9659


## Without Title Duplicate Count - Cosine Similarity

In [ ]:
# Hybrid BERT + Numeric + Region Embedding
import pandas as pd
import numpy as np
import torch
import time
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel

# ================== LOAD & PREPROCESS DATA ==================
# Load dataset
df = pd.read_csv('/content/drive/MyDrive/Research Important Datasets/Final_with_caption_similarity_50k_cosine.csv')
df = df.dropna(subset=['title'])

# Fill missing values
numeric_cols = ['num_comments', 'score', 'upvote_ratio', 'polarity', 'emotion_score', 'title_dup_count', 'image_caption_similarity']
df[numeric_cols] = df[numeric_cols].fillna(0)
df['sub_region'] = df['sub_region'].fillna('unknown')

# Region ID encoding
df['region_id'] = df['sub_region'].astype('category').cat.codes
region2id = dict(enumerate(df['sub_region'].astype('category').cat.categories))
num_regions = len(region2id)

# Final feature sets
all_numeric = numeric_cols
X_text = df['title'].values
X_numeric = df[all_numeric].values
X_region = df['region_id'].values
y = df['2_way_label'].values

# ================== SPLIT DATA ==================
X_text_tv, X_text_test, X_num_tv, X_num_test, X_reg_tv, X_reg_test, y_tv, y_test = train_test_split(
    X_text, X_numeric, X_region, y, test_size=0.15, stratify=y, random_state=42
)
X_text_train, X_text_val, X_num_train, X_num_val, X_reg_train, X_reg_val, y_train, y_val = train_test_split(
    X_text_tv, X_num_tv, X_reg_tv, y_tv, test_size=0.1765, stratify=y_tv, random_state=42
)

# Normalize numeric features
scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_num_train)
X_num_val = scaler.transform(X_num_val)
X_num_test = scaler.transform(X_num_test)

# ================== TOKENIZER ==================
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# ================== CUSTOM DATASET ==================
class FakeNewsDataset(Dataset):
    def __init__(self, texts, numerics, region_ids, labels):
        self.texts = texts
        self.numerics = numerics
        self.region_ids = region_ids
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=32,
            return_tensors="pt"
        )
        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'numerics': torch.tensor(self.numerics[idx], dtype=torch.float32),
            'region_id': torch.tensor(self.region_ids[idx], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# ================== DATALOADERS ==================
train_dataset = FakeNewsDataset(X_text_train, X_num_train, X_reg_train, y_train)
val_dataset   = FakeNewsDataset(X_text_val, X_num_val, X_reg_val, y_val)
test_dataset  = FakeNewsDataset(X_text_test, X_num_test, X_reg_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64)
test_loader  = DataLoader(test_dataset, batch_size=64)

# ================== MODEL ==================
class HybridBERTRegionEmbeddingModel(nn.Module):
    def __init__(self, numeric_input_dim, num_regions, region_embed_dim=16):
        super(HybridBERTRegionEmbeddingModel, self).__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.region_embedding = nn.Embedding(num_regions, region_embed_dim)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(768 + numeric_input_dim + region_embed_dim, 128)
        self.fc2 = nn.Linear(128, 1)
        self.relu = nn.ReLU()

    def forward(self, input_ids, attention_mask, numerics, region_ids):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        region_embeds = self.region_embedding(region_ids)
        combined = torch.cat((cls_output, numerics, region_embeds), dim=1)
        x = self.relu(self.fc1(self.dropout(combined)))
        return torch.sigmoid(self.fc2(x))

# ================== TRAINING ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridBERTRegionEmbeddingModel(
    numeric_input_dim=X_num_train.shape[1],
    num_regions=num_regions
).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)
start_time = time.time()

for epoch in range(3):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        numerics = batch['numerics'].to(device)
        region_ids = batch['region_id'].to(device)
        labels = batch['label'].to(device).unsqueeze(1)

        outputs = model(input_ids, attention_mask, numerics, region_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {epoch_loss:.4f}")

end_time = time.time()
print(f"\nTotal training time: {end_time - start_time:.2f} seconds")

# ================== EVALUATION ==================
def evaluate_model(dataloader, dataset_name="Validation"):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            numerics = batch['numerics'].to(device)
            region_ids = batch['region_id'].to(device)
            labels = batch['label'].to(device).unsqueeze(1)

            outputs = model(input_ids, attention_mask, numerics, region_ids)
            probs = outputs.cpu().numpy()
            preds = (probs > 0.5).astype(int)

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)

    print(f"\n{dataset_name} Classification Report:")
    print(classification_report(all_labels, all_preds))
    if np.isnan(all_probs).any():
        print(f"Warning: NaNs in {dataset_name} probabilities. Skipping AUC.")
    else:
        auc_score = roc_auc_score(all_labels, all_probs)
        print(f"{dataset_name} AUC: {auc_score:.4f}")

# Evaluate
evaluate_model(val_loader, "Validation")
evaluate_model(test_loader, "Test")

## Not Checking Similarity but use both Title and Image Vectors

In [2]:
# 0. INSTALL DEPENDENCIES IF NEEDED
# !pip install transformers

# 1. IMPORTS
import pandas as pd
import numpy as np
import torch
import time
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
from transformers import BertTokenizer, BertModel

# 2. LOAD DATA
df = pd.read_csv('/content/drive/MyDrive/Research Important Datasets/Final/finalDataset.csv')
df = df.dropna(subset=['title'])

# Handle missing values
df['caption'] = df['caption'].fillna("")
df['sub_region'] = df['sub_region'].fillna('unknown')
numeric_cols = ['num_comments', 'score', 'upvote_ratio', 'polarity', 'emotion_score', 'title_dup_count']
df[numeric_cols] = df[numeric_cols].fillna(0)

# Region encoding
df['region_id'] = df['sub_region'].astype('category').cat.codes
region2id = dict(enumerate(df['sub_region'].astype('category').cat.categories))
num_regions = len(region2id)

# Features
X_title   = df['title'].values
X_caption = df['caption'].values
X_numeric = df[numeric_cols].values
X_region  = df['region_id'].values
y         = df['2_way_label'].values

# 3. SPLIT DATA
X_title_tv, X_title_test, X_caption_tv, X_caption_test, X_num_tv, X_num_test, X_reg_tv, X_reg_test, y_tv, y_test = train_test_split(
    X_title, X_caption, X_numeric, X_region, y, test_size=0.15, stratify=y, random_state=42
)
X_title_train, X_title_val, X_caption_train, X_caption_val, X_num_train, X_num_val, X_reg_train, X_reg_val, y_train, y_val = train_test_split(
    X_title_tv, X_caption_tv, X_num_tv, X_reg_tv, y_tv, test_size=0.1765, stratify=y_tv, random_state=42
)

# Normalize numeric features
scaler = StandardScaler()
X_num_train = scaler.fit_transform(X_num_train)
X_num_val   = scaler.transform(X_num_val)
X_num_test  = scaler.transform(X_num_test)

# 4. TOKENIZER
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# 5. CUSTOM DATASET
class DualInputDataset(Dataset):
    def __init__(self, titles, captions, numerics, region_ids, labels):
        self.titles = titles
        self.captions = captions
        self.numerics = numerics
        self.region_ids = region_ids
        self.labels = labels

    def __len__(self):
        return len(self.titles)

    def __getitem__(self, idx):
        title_tokens = tokenizer(
            self.titles[idx], padding='max_length', truncation=True, max_length=32, return_tensors="pt"
        )
        caption_tokens = tokenizer(
            self.captions[idx], padding='max_length', truncation=True, max_length=32, return_tensors="pt"
        )
        return {
            'input_ids_title': title_tokens['input_ids'].squeeze(0),
            'attention_mask_title': title_tokens['attention_mask'].squeeze(0),
            'input_ids_caption': caption_tokens['input_ids'].squeeze(0),
            'attention_mask_caption': caption_tokens['attention_mask'].squeeze(0),
            'numerics': torch.tensor(self.numerics[idx], dtype=torch.float32),
            'region_id': torch.tensor(self.region_ids[idx], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.float32)
        }

# 6. DATALOADERS
train_loader = DataLoader(DualInputDataset(X_title_train, X_caption_train, X_num_train, X_reg_train, y_train), batch_size=64, shuffle=True)
val_loader   = DataLoader(DualInputDataset(X_title_val, X_caption_val, X_num_val, X_reg_val, y_val), batch_size=64)
test_loader  = DataLoader(DualInputDataset(X_title_test, X_caption_test, X_num_test, X_reg_test, y_test), batch_size=64)

# 7. MODEL
class DualBERTModel(nn.Module):
    def __init__(self, numeric_input_dim, num_regions, region_embed_dim=16):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.region_embedding = nn.Embedding(num_regions, region_embed_dim)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(768*2 + numeric_input_dim + region_embed_dim, 128)
        self.fc2 = nn.Linear(128, 1)
        self.relu = nn.ReLU()

    def forward(self, input_ids_title, attention_mask_title, input_ids_caption, attention_mask_caption, numerics, region_ids):
        cls_title   = self.bert(input_ids=input_ids_title, attention_mask=attention_mask_title).last_hidden_state[:, 0, :]
        cls_caption = self.bert(input_ids=input_ids_caption, attention_mask=attention_mask_caption).last_hidden_state[:, 0, :]
        region_embeds = self.region_embedding(region_ids)
        combined = torch.cat((cls_title, cls_caption, numerics, region_embeds), dim=1)
        x = self.relu(self.fc1(self.dropout(combined)))
        return torch.sigmoid(self.fc2(x))

# 8. TRAINING SETUP
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DualBERTModel(X_num_train.shape[1], num_regions).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-5)

# 9. TRAIN LOOP
for epoch in range(3):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids_title = batch['input_ids_title'].to(device)
        attention_mask_title = batch['attention_mask_title'].to(device)
        input_ids_caption = batch['input_ids_caption'].to(device)
        attention_mask_caption = batch['attention_mask_caption'].to(device)
        numerics = batch['numerics'].to(device)
        region_ids = batch['region_id'].to(device)
        labels = batch['label'].to(device).unsqueeze(1)

        outputs = model(input_ids_title, attention_mask_title, input_ids_caption, attention_mask_caption, numerics, region_ids)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

# 10. EVALUATION
def evaluate_model(dataloader, name):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in dataloader:
            input_ids_title = batch['input_ids_title'].to(device)
            attention_mask_title = batch['attention_mask_title'].to(device)
            input_ids_caption = batch['input_ids_caption'].to(device)
            attention_mask_caption = batch['attention_mask_caption'].to(device)
            numerics = batch['numerics'].to(device)
            region_ids = batch['region_id'].to(device)
            labels = batch['label'].to(device).unsqueeze(1)

            outputs = model(input_ids_title, attention_mask_title, input_ids_caption, attention_mask_caption, numerics, region_ids)
            probs = outputs.cpu().numpy()
            preds = (probs > 0.5).astype(int)

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs)

    print(f"\n{name} Classification Report:")
    print(classification_report(all_labels, all_preds))
    auc = roc_auc_score(all_labels, all_probs)
    print(f"{name} AUC: {auc:.4f}")

# Evaluate
evaluate_model(val_loader, "Validation")
evaluate_model(test_loader, "Test")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Epoch 1 Loss: 299.0404
Epoch 2 Loss: 204.2748
Epoch 3 Loss: 127.0237

Validation Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.87      0.89      6377
         1.0       0.91      0.94      0.92      8626

    accuracy                           0.91     15003
   macro avg       0.91      0.91      0.91     15003
weighted avg       0.91      0.91      0.91     15003

Validation AUC: 0.9693

Test Classification Report:
              precision    recall  f1-score   support

         0.0       0.92      0.87      0.90      6375
         1.0       0.91      0.95      0.93      8625

    accuracy                           0.92     15000
   macro avg       0.92      0.91      0.91     15000
weighted avg       0.92      0.92      0.91     15000

Test AUC: 0.9696
